In [1]:
import pandas as pd

In [28]:
features = pd.read_parquet('data/diabetes_features.parquet')
target = pd.read_parquet('data/diabetes_target.parquet')

# Columns and Data Types

In [3]:
print(features.info())

<class 'pandas.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 49 columns):
 #   Column                    Non-Null Count   Dtype   
---  ------                    --------------   -----   
 0   encounter_id              101766 non-null  int64   
 1   patient_nbr               101766 non-null  int64   
 2   race                      101766 non-null  category
 3   gender                    101766 non-null  category
 4   age                       101766 non-null  category
 5   weight                    101766 non-null  category
 6   admission_type_id         101766 non-null  int64   
 7   discharge_disposition_id  101766 non-null  int64   
 8   admission_source_id       101766 non-null  int64   
 9   time_in_hospital          101766 non-null  int64   
 10  payer_code                101766 non-null  category
 11  medical_specialty         101766 non-null  category
 12  num_lab_procedures        101766 non-null  int64   
 13  num_procedures            101766 non-nul

In [4]:
for c in features.columns.values:
    print(c)

encounter_id
patient_nbr
race
gender
age
weight
admission_type_id
discharge_disposition_id
admission_source_id
time_in_hospital
payer_code
medical_specialty
num_lab_procedures
num_procedures
num_medications
number_outpatient
number_emergency
number_inpatient
diag_1
diag_2
diag_3
number_diagnoses
max_glu_serum
A1Cresult
metformin
repaglinide
nateglinide
chlorpropamide
glimepiride
acetohexamide
glipizide
glyburide
tolbutamide
pioglitazone
rosiglitazone
acarbose
miglitol
troglitazone
tolazamide
examide
citoglipton
insulin
glyburide.metformin
glipizide.metformin
glimepiride.pioglitazone
metformin.rosiglitazone
metformin.pioglitazone
change
diabetesMed


In [5]:
features.patient_nbr.value_counts()

patient_nbr
88785891     40
43140906     28
1660293      23
23199021     23
88227540     23
             ..
183087545     1
188574944     1
140199494     1
120975314     1
175429310     1
Name: count, Length: 71518, dtype: int64

In [6]:
# Exploring a single patient
features[features.patient_nbr == 88785891]

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,examide,citoglipton,insulin,glyburide.metformin,glipizide.metformin,glimepiride.pioglitazone,metformin.rosiglitazone,metformin.pioglitazone,change,diabetesMed
38307,119039172,88785891,Caucasian,Female,[20-30),?,1,1,7,1,...,No,No,Up,No,No,No,No,No,Ch,Yes
40252,125094312,88785891,Caucasian,Female,[20-30),?,1,1,7,1,...,No,No,Down,No,No,No,No,No,Ch,Yes
40661,126171582,88785891,Caucasian,Female,[20-30),?,1,1,7,5,...,No,No,Up,No,No,No,No,No,Ch,Yes
44515,137245596,88785891,Caucasian,Female,[20-30),?,3,1,7,2,...,No,No,Up,No,No,No,No,No,Ch,Yes
45147,139425576,88785891,Caucasian,Female,[20-30),?,1,1,7,2,...,No,No,Up,No,No,No,No,No,Ch,Yes
45986,141994242,88785891,Caucasian,Female,[20-30),?,2,1,7,4,...,No,No,Up,No,No,No,No,No,Ch,Yes
50167,150986298,88785891,Caucasian,Female,[20-30),?,2,1,7,1,...,No,No,Up,No,No,No,No,No,Ch,Yes
50393,151413846,88785891,Caucasian,Female,[20-30),?,1,1,7,4,...,No,No,Up,No,No,No,No,No,Ch,Yes
50773,152188656,88785891,Caucasian,Female,[20-30),?,2,7,7,1,...,No,No,Up,No,No,No,No,No,Ch,Yes
51519,153558456,88785891,Caucasian,Female,[20-30),?,2,1,7,1,...,No,No,Up,No,No,No,No,No,Ch,Yes


# Missing Data

#### Key Findings

- Weight is missing for an entire patient, except for 36 patients captured below
- Missing values are encoded in some columns using "?" and None

In [7]:
# Diagnosing if weight is always missing for a patient
weight_diagnostic = features.loc[:, ['patient_nbr', 'weight']]

weight_diagnostic['missing_weight'] = weight_diagnostic['weight'].apply(lambda x: 1 if x == "?" else 0)

# Group and aggregate
grouped = weight_diagnostic.groupby('patient_nbr').agg({
    'missing_weight': 'sum',  # count of missing weights
    'weight': 'count'          # total records for that patient
})

# Patients where missing_weight == total records are always missing
grouped['always_missing'] = grouped['missing_weight'] == grouped['weight']

grouped[(grouped["missing_weight"] != grouped['weight']) & (grouped["missing_weight"] > 0)]

,missing_weight,weight,always_missing
patient_nbr,,,
1743804,1,2,False
3515238,1,2,False
4019499,1,2,False
4790016,1,8,False
6316542,1,2,False
7994817,1,2,False
21684645,1,2,False
30775878,1,2,False
35879364,1,2,False


In [8]:
# Inspecting one patient that has mixed missing data
features[features.patient_nbr == 4790016]

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,examide,citoglipton,insulin,glyburide.metformin,glipizide.metformin,glimepiride.pioglitazone,metformin.rosiglitazone,metformin.pioglitazone,change,diabetesMed
18056,65754690,4790016,Caucasian,Female,[20-30),[50-75),2,1,1,1,...,No,No,No,No,No,No,No,No,No,No
23306,79654248,4790016,Caucasian,Female,[20-30),[50-75),1,1,7,7,...,No,No,Up,No,No,No,No,No,Ch,Yes
24608,83243016,4790016,Caucasian,Female,[20-30),?,2,18,4,1,...,No,No,No,No,No,No,No,No,No,No
28426,93131496,4790016,Caucasian,Female,[20-30),[75-100),1,5,6,3,...,No,No,Up,No,No,No,No,No,Ch,Yes
32564,103670874,4790016,Caucasian,Female,[20-30),[50-75),1,1,7,1,...,No,No,Steady,No,No,No,No,No,No,Yes
49008,148883136,4790016,Caucasian,Female,[20-30),[50-75),1,1,7,7,...,No,No,No,No,No,No,No,No,No,No
61923,172669098,4790016,Caucasian,Female,[20-30),[50-75),1,1,7,3,...,No,No,Up,No,No,No,No,No,Ch,Yes
74378,221409540,4790016,Caucasian,Female,[20-30),[50-75),1,1,7,1,...,No,No,Up,No,No,No,No,No,Ch,Yes


### Diving into the use of Identifiers ("?", None) for missing data

In [9]:
qmark_missing = (features == "?").sum()
print(qmark_missing[qmark_missing > 0])

race                  2273
weight               98569
payer_code           40256
medical_specialty    49949
diag_1                  21
diag_2                 358
diag_3                1423
dtype: int64


In [10]:
none_missing = (features == "None").sum()
print(none_missing[none_missing > 0])

max_glu_serum    96420
A1Cresult        84748
dtype: int64


#### Correlation of Missing Values

In [29]:
((features == "?") | (features == "None")).astype(int).corr().round(4)

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,examide,citoglipton,insulin,glyburide.metformin,glipizide.metformin,glimepiride.pioglitazone,metformin.rosiglitazone,metformin.pioglitazone,change,diabetesMed
encounter_id,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
patient_nbr,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
race,NaN,NaN,1.0000,NaN,NaN,-0.0254,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
gender,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
age,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
weight,NaN,NaN,-0.0254,NaN,NaN,1.0000,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
admission_type_id,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
discharge_disposition_id,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
admission_source_id,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
time_in_hospital,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


#### Exploring Patterns in Missing Data

Missing data in the following columns depends on admission type and admission source:

- max_glu_serum
- A1Cresult
- weight
- payer_code
- medical_specialty

This pattern suggests data is MAR with dependency on the admission source and type. A Preliminary approach is to fit the model after subsetting by admission type and source, or picking a model (i.e. tree-based) that will automatically subset the data

In [ ]:
# Admission Type
for f in ['max_glu_serum', 'A1Cresult', 'weight', 'payer_code', 'medical_specialty']:
    ct = pd.crosstab(features['admission_type_id'], features[f])
    print(ct.apply(lambda x: x / sum(x), axis = 1))


max_glu_serum          >200      >300      None      Norm
admission_type_id                                        
1                  0.002704  0.006186  0.984182  0.006927
2                  0.002652  0.002110  0.988366  0.006872
3                  0.000053  0.000318  0.998781  0.000848
4                  0.000000  0.000000  1.000000  0.000000
5                  0.173250  0.116405  0.422571  0.287774
6                  0.086940  0.061992  0.718201  0.132867
7                  0.000000  0.000000  1.000000  0.000000
8                  0.000000  0.000000  1.000000  0.000000
A1Cresult                >7        >8      None      Norm
admission_type_id                                        
1                  0.040415  0.093073  0.807964  0.058548
2                  0.044048  0.077597  0.827219  0.051136
3                  0.025704  0.045736  0.897557  0.031003
4                  0.200000  0.000000  0.800000  0.000000
5                  0.026332  0.047022  0.907001  0.019645
6             

In [47]:
# Admission Source
for f in ['max_glu_serum', 'A1Cresult', 'weight', 'payer_code', 'medical_specialty']:
    ct = pd.crosstab(features['admission_source_id'], features[f])
    print(ct.apply(lambda x: x / sum(x), axis = 1))

max_glu_serum            >200      >300      None      Norm
admission_source_id                                        
1                    0.001657  0.001421  0.992559  0.004363
2                    0.006341  0.001812  0.980978  0.010870
3                    0.000000  0.000000  1.000000  0.000000
4                    0.000000  0.000314  0.999686  0.000000
5                    0.000000  0.000000  1.000000  0.000000
6                    0.000000  0.000000  1.000000  0.000000
7                    0.005079  0.008418  0.978833  0.007670
8                    0.000000  0.000000  1.000000  0.000000
9                    0.000000  0.000000  1.000000  0.000000
10                   0.000000  0.000000  1.000000  0.000000
11                   0.000000  0.000000  1.000000  0.000000
13                   0.000000  0.000000  1.000000  0.000000
14                   0.000000  0.000000  1.000000  0.000000
17                   0.167674  0.108391  0.426781  0.297154
20                   0.000000  0.000000 

# Outliers